# 04.3.4 - Supervised Learning: Klasifikasi Akurasi Tertinggi dengan ANN (K-Means DE)

## 1. Tujuan Analisis
Sebagai penutup dari seluruh rangkaian eksperimen kita, kita akan mengerahkan algoritma paling mutakhir yaitu **Artificial Neural Network (ANN)** untuk membedah data hasil **K-Means DE**.

Kita kembali menggunakan konsep "otak tiruan" (atau pabrik dengan banyak lapisan pegawai spesialis di ruang tengah). Tujuannya adalah untuk melihat apakah model yang sangat kompleks ini mampu meraih akurasi tebakan yang mendekati sempurna pada pengelompokan pelanggan versi DE.

## 2. Kapasitas Komputasi
Mengingat total dataset yang kita miliki sangat ramah untuk diproses (berkisar di angka 4.000-an baris), kita akan menggunakan 100% baris data untuk proses pelatihan (*training*). Ini memberikan kesempatan bagi ANN untuk mempelajari setiap detail karakteristik pelanggan tanpa ada informasi yang terpotong.

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: DE)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-de.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% Train, 20% Test (Menggunakan 100% baris data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Proses Scaling Data
Langkah *Scaling* (Penyetaraan Skala) wajib dilakukan. Jika tidak disetarakan, "pegawai" di dalam jaringan saraf tiruan ini akan kebingungan dan secara keliru menganggap bahwa fitur dengan angka jutaan (seperti nominal transaksi) jauh lebih penting daripada fitur yang angkanya kecil. Kita menggunakan `StandardScaler` agar semuanya dievaluasi secara adil.

In [2]:
# 2. SCALING DATA (Langkah wajib untuk ANN)
scaler_ann = StandardScaler()

# Model scaler belajar dari data latih (Train) sekaligus menyetarakan angkanya
X_train_scaled = scaler_ann.fit_transform(X_train)

# Data ujian (Test) juga disetarakan agar formatnya seragam
X_test_scaled = scaler_ann.transform(X_test)

## 4. Training Model ANN
Kita menyewa 100 "pegawai spesialis" (`hidden_layer_sizes=(100,)`) dan memberikan waktu belajar maksimal sebanyak 500 putaran (`max_iter=500`). 

*Catatan:* Peringatan `ConvergenceWarning` (kotak merah/pink) yang mungkin muncul saat kode dijalankan bukanlah sebuah *error*. Itu sekadar informasi bahwa mesin merasa masih sanggup belajar lebih lama dari batas 500 putaran yang kita berikan, namun prosesnya dihentikan. Hasil yang didapatkan pun sudah sangat optimal.

In [3]:
# 3. TRAINING MODEL ANN
# Menggunakan max_iter=500 agar model punya cukup waktu belajar
model_ann = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
model_ann.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data ujian
prediksi_ann = model_ann.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: ANN (DE K-MEANS) ===\n")
print(classification_report(y_test, prediksi_ann))

=== CLASSIFICATION REPORT: ANN (DE K-MEANS) ===

              precision    recall  f1-score   support

           0       0.98      0.95      0.96       138
           1       0.99      0.98      0.98       165
           2       0.95      0.99      0.97       165
           3       0.99      0.96      0.98        84
           4       0.96      0.96      0.96       111
           5       0.97      0.97      0.97       204

    accuracy                           0.97       867
   macro avg       0.97      0.97      0.97       867
weighted avg       0.97      0.97      0.97       867



c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### Analisis Performa
Luar biasa! Rapor performa menunjukkan bahwa otak tiruan ANN berhasil meraih akurasi sebesar **97%** (atau `0.97`) pada dataset DE. 

Seluruh kelas pelanggan berhasil ditebak dengan presisi dan *recall* di atas 95%. Angka ini menempatkan ANN sebagai jawara mutlak dalam hal akurasi, membuktikan bahwa kompleksitas algoritmanya sangat sepadan dengan hasil tebakannya yang nyaris tanpa cela.

## 5. Ekspor Model dan Scaler
Kita simpan alat penyetara angka dan mesin otak tiruannya dengan akhiran `_de`. Kedua *file* inilah yang nantinya akan bekerja di belakang layar untuk memproses data baru.

In [4]:
# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan Scaler
joblib.dump(scaler_ann, '../models/scaler_ann_de.pkl')

# Simpan Model ANN
joblib.dump(model_ann, '../models/model_ann_classification_de.pkl')

print("\n[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!")


[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!
